# 06 - Machine Learning: Predicción de Resultados

En este notebook se cargan partidos históricos de la Premier League desde una base SQLite, se construyen variables derivadas y se entrenan modelos de clasificación para predecir el resultado del partido (victoria local, empate o victoria visitante) a partir de las características definidas.

In [ ]:
import sqlite3
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

## 1. Carga de datos

In [ ]:
ruta_db = "../data/premier_league.db"
conexion = sqlite3.connect(ruta_db)

consulta = "SELECT id, fecha, equipo_local, equipo_visitante, goles_local, goles_visitante FROM partidos"
df = pd.read_sql_query(consulta, conexion)
conexion.close()

print("Shape del DataFrame:", df.shape)
print(df.head())

## 2. Feature Engineering

In [ ]:
def etiquetar_resultado(fila):
    if fila["goles_local"] > fila["goles_visitante"]:
        return "Local"
    if fila["goles_local"] == fila["goles_visitante"]:
        return "Empate"
    return "Visitante"


df["resultado"] = df.apply(etiquetar_resultado, axis=1)
df["diferencia_goles"] = df["goles_local"] - df["goles_visitante"]
df["total_goles"] = df["goles_local"] + df["goles_visitante"]

print("Distribución de resultado:")
print(df["resultado"].value_counts())

## 3. Preparación para ML

In [ ]:
X = df[["goles_local", "goles_visitante", "diferencia_goles", "total_goles"]]
y = df["resultado"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Tamaño conjunto de entrenamiento: X_train {X_train.shape}, y_train {y_train.shape}")
print(f"Tamaño conjunto de prueba: X_test {X_test.shape}, y_test {y_test.shape}")

## 4. Modelos

In [ ]:
modelo_lr = LogisticRegression(max_iter=1000)
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)

modelo_lr.fit(X_train, y_train)
modelo_rf.fit(X_train, y_train)

y_pred_lr = modelo_lr.predict(X_test)
y_pred_rf = modelo_rf.predict(X_test)

for nombre, y_pred in [("Regresión logística", y_pred_lr), ("Bosque aleatorio", y_pred_rf)]:
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted")
    rec = recall_score(y_test, y_pred, average="weighted")
    print(f"\n{nombre}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precisión (ponderada): {prec:.4f}")
    print(f"  Recall (ponderado):    {rec:.4f}")

## 5. Evaluación

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

etiquetas_clase = sorted(y.unique())

cm_lr = confusion_matrix(y_test, y_pred_lr, labels=etiquetas_clase)
disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=etiquetas_clase)
disp_lr.plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Matriz de confusión — Regresión logística")

cm_rf = confusion_matrix(y_test, y_pred_rf, labels=etiquetas_clase)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=etiquetas_clase)
disp_rf.plot(ax=axes[1], cmap="Greens", colorbar=False)
axes[1].set_title("Matriz de confusión — Bosque aleatorio")

plt.tight_layout()
plt.show()

print("Informe de clasificación — Regresión logística")
print(classification_report(y_test, y_pred_lr))
print("Informe de clasificación — Bosque aleatorio")
print(classification_report(y_test, y_pred_rf))

## 6. Conclusiones

[Texto placeholder] Se completó el flujo de carga, ingeniería de variables, entrenamiento y evaluación de dos modelos. Aquí se resumirían las métricas obtenidas, la comparación entre regresión logística y bosque aleatorio, y posibles mejoras (por ejemplo, incorporar estadísticas previas al partido en lugar de goles finales si el objetivo es predicción *a priori*).